# Day 7 · Case study: reproducing Tabin et al. (2023), Figure 7
*Measuring Manuscripts*

This notebook reproduces Figure 7 from Tabin et al. (2023), *Collaborative Annotation and Computational Analysis of Hieratic*. The figure takes one hieratic sign, *G17* (the owl, 𓅓), as written in two Middle Egyptian texts, the *Shipwrecked Sailor* and the *Eloquent Peasant* (manuscript B1). It turns every drawn instance into a row of numbers and projects them to two dimensions with PCA, the same move as the main Day 7 notebook. Each token is a dot, colored by its text.

The point is that the owl was not drawn one way. PCA pulls apart distinct *forms* of the sign. Some forms are shared across both texts, and one is found almost only in the Shipwrecked Sailor.

It reads the Isut sign images straight from disk, from the `Isut/` folder beside this notebook. We will rearrange the data for online use later. For now, run this from the Day 7 folder.

## 1. Setup

In [ ]:
import os, csv
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

DATA  = 'Isut'                                  # the data folder beside this notebook
G17   = 'U+13153'                               # Gardiner G17, the owl 𓅓
TEXTS = ['Peasant B1', 'Shipwrecked']           # display names
FOLDER = {'Peasant B1': 'PeasantB1', 'Shipwrecked': 'Shipwrecked'}

def img_path(rel):
    # the label tables were written before the folder was renamed to 'Isut', so they
    # still say 'texts/...'. Strip that and point at the real folder.
    if rel.startswith('texts/'):
        rel = rel[len('texts/'):]
    return os.path.join(DATA, rel)

## 2. Load one sign from each text

Each text folder carries a table, `overlay_output/all_colored_signs.csv`, that labels every annotated glyph with its Unicode codepoint and the path to its cropped image. We keep only the rows whose codepoint is G17, load each little picture, square it up so its shape is preserved, and flatten it into a row of numbers. This is the *image to feature vector* step from the main Day 7 work, applied to real hieratic crops.

In [ ]:
def load_sign(text_folder, unicode_key, size=48):
    csv_path = os.path.join(DATA, text_folder, 'overlay_output', 'all_colored_signs.csv')
    rows = [r for r in csv.DictReader(open(csv_path, encoding='utf-8'))
            if r['unicode_key'] == unicode_key]
    vectors, crops = [], []
    for r in rows:
        im   = Image.open(img_path(r['image_path'])).convert('RGBA')
        flat = Image.alpha_composite(Image.new('RGBA', im.size, (255, 255, 255, 255)), im)
        gray = flat.convert('L')
        w, h = gray.size                          # pad to a square so the proportions survive
        s = max(w, h)
        square = Image.new('L', (s, s), 255)
        square.paste(gray, ((s - w) // 2, (s - h) // 2))
        small = square.resize((size, size))
        crops.append(small)
        vectors.append(255.0 - np.asarray(small, dtype=float))   # invert so ink is large
    return np.array([v.ravel() for v in vectors]) / 255.0, crops

X_parts, crops, labels = [], [], []
for name in TEXTS:
    V, C = load_sign(FOLDER[name], G17)
    X_parts.append(V); crops += C; labels += [name] * len(V)
    print(f'{name}: {len(V)} instances of G17')

X = np.vstack(X_parts)
labels = np.array(labels)
print('feature matrix:', X.shape)

## 3. A look at the glyphs

Before any math, here are a few of the owls from each text. Even by eye, they are not all the same shape.

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(11, 2.8))
for row, name in enumerate(TEXTS):
    idx = [i for i, l in enumerate(labels) if l == name][:10]
    for col, ax in enumerate(axes[row]):
        ax.axis('off')
        if col < len(idx):
            ax.imshow(crops[idx[col]], cmap='gray')
    axes[row, 0].set_title(name, loc='left', fontsize=9)
plt.tight_layout(); plt.show()

## 4. Reproducing Figure 7

Now the projection. PCA finds the two directions of greatest variation among the owl shapes and plots every token on those two axes. Color marks the text. The thumbnails at the edges show the actual glyph at the most extreme points, the way Isut shows a sign's shape when you hover over its dot. The owls at opposite ends of the plot are visibly different forms.

In [ ]:
coords = PCA(n_components=2).fit_transform(X - X.mean(0))

fig, ax = plt.subplots(figsize=(8, 7))
color = {'Shipwrecked': '#7a2630', 'Peasant B1': '#27408b'}
for name in TEXTS:
    m = labels == name
    ax.scatter(coords[m, 0], coords[m, 1], c=color[name], s=26, alpha=0.75,
               edgecolors='none', label=name)

# drop the real glyph onto the four most extreme points
extremes = {coords[:, 0].argmin(), coords[:, 0].argmax(),
            coords[:, 1].argmin(), coords[:, 1].argmax()}
for i in extremes:
    box = AnnotationBbox(OffsetImage(np.asarray(crops[i]), cmap='gray', zoom=0.7),
                         (coords[i, 0], coords[i, 1]), frameon=True, pad=0.1)
    ax.add_artist(box)

ax.legend(); ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('G17 (the owl) across two texts, PCA to 2D')
plt.tight_layout(); plt.show()

## 5. The three forms

The paper reads three types out of this plot. We can make that quantitative with the clustering from earlier in the day. Run *k-means* for three groups on the PCA coordinates, then for each group show its most central owl and how the two texts split inside it. One group comes out heavily Shipwrecked, matching the paper's reading that one form is found almost only there.

In [ ]:
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(coords)

fig, axes = plt.subplots(1, 3, figsize=(7.5, 2.8))
print('Three forms of the owl, by k-means on the PCA coordinates:')
for c in range(3):
    m = km.labels_ == c
    total = int(m.sum())
    ship  = int(((labels == 'Shipwrecked') & m).sum())
    peas  = total - ship
    centre = coords[m].mean(0)                                  # most central token = medoid
    medoid = np.where(m)[0][np.argmin(((coords[m] - centre) ** 2).sum(1))]
    axes[c].imshow(crops[medoid], cmap='gray'); axes[c].axis('off')
    axes[c].set_title(f'form {c}\n{ship/total:.0%} Shipwrecked', fontsize=9)
    print(f'  form {c}: {total:3} tokens  ->  Shipwrecked {ship:3} ({ship/total:.0%}), '
          f'Peasant B1 {peas:3} ({peas/total:.0%})')
plt.tight_layout(); plt.show()

## 6. Reading the picture

This is the case study from the end of the Tabin paper, rebuilt from the data on disk. A single sign, measured across two texts, splits into a few recurring forms. Two of them appear in both manuscripts, and one is almost confined to the Shipwrecked Sailor. That is the kind of cross-textual observation the main Day 7 tools are for, here on real hieratic rather than toy data. The same code would run on any other sign by changing one line: set `G17` to another codepoint and reload.

> 🔧 *TO BUILD*: this reads the images from the local `Isut/` folder. To run it in Colab we will host the crops and label tables online and point the loader at those URLs, the same move as the Day 2 and Day 3 notebooks. The analysis above does not change.